# Model Training & Comparison

This notebook trains multiple machine learning models (Linear Regression, Random Forest, XGBoost) to predict Philippine crop production volume.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

## 1. Data Preprocessing
Load data, reshape from wide to long (quarters only), extract Year/Quarter, handle NaNs, encode categorical variables.

In [ ]:
file_path = r'c:\Users\James\Music\CVPP\data\original\Major Crops- Volume of Production, by Region, Quarter, and Semester, 2010-2026.csv'
df = pd.read_csv(file_path, header=1)

# Ensure 'Region' and 'Crop' are properly identified if they aren't explicitly named. Assuming first two cols are Crop and Region based on dataset structure.
df.rename(columns={df.columns[0]: 'Crop', df.columns[1]: 'Region'}, inplace=True)

# Forward fill Crop if it's hierarchical, though based on Philippine PSA data, it's often hierarchical.
df['Crop'] = df['Crop'].fillna(method='ffill')

# Keep only Quarter columns
quarter_cols = [c for c in df.columns if 'Quarter' in str(c) and 'Semester' not in str(c) and 'Annual' not in str(c)]
id_cols = ['Crop', 'Region']
df_filtered = df[id_cols + quarter_cols].copy()

# Melt
df_long = pd.melt(df_filtered, id_vars=id_cols, value_vars=quarter_cols, var_name='Time', value_name='Volume')

# Extract Year and Quarter
# Format typically: '2010 Quarter1'
df_long[['Year', 'Quarter']] = df_long['Time'].str.extract(r'(\d{4})\s*Quarter(\d)')
df_long['Year'] = pd.to_numeric(df_long['Year'])
df_long['Quarter'] = pd.to_numeric(df_long['Quarter'])

# Convert Volume to numeric, coercing errors like '..'
df_long['Volume'] = pd.to_numeric(df_long['Volume'].astype(str).str.replace(',', ''), errors='coerce')
df_long.dropna(subset=['Volume'], inplace=True)

# Filter out 'PHILIPPINES' rows and clean region names
df_long = df_long[~df_long['Region'].str.upper().str.contains('PHILIPPINES')].copy()
df_long['Region'] = df_long['Region'].astype(str).str.lstrip('. ')

# Label Encoding
le_crop = LabelEncoder()
le_region = LabelEncoder()
df_long['Crop_Encoded'] = le_crop.fit_transform(df_long['Crop'])
df_long['Region_Encoded'] = le_region.fit_transform(df_long['Region'])

# Train-test split
features = ['Crop_Encoded', 'Region_Encoded', 'Year', 'Quarter']
X = df_long[features]
y = df_long['Volume']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f'Train shape: {X_train.shape}, Test shape: {X_test.shape}')

## 2. Model 1: Linear Regression

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_preds = lr_model.predict(X_test)

lr_r2 = r2_score(y_test, lr_preds)
lr_mae = mean_absolute_error(y_test, lr_preds)
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))

print(f'Linear Regression - R2: {lr_r2:.4f}, MAE: {lr_mae:.4f}, RMSE: {lr_rmse:.4f}')

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.scatterplot(x=y_test, y=lr_preds, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.title('LR: Predicted vs Actual')
plt.xlabel('Actual Volume')
plt.ylabel('Predicted Volume')

plt.subplot(1, 2, 2)
sns.histplot(y_test - lr_preds, bins=50, kde=True)
plt.title('LR: Residual Distribution')
plt.xlabel('Residual')
plt.tight_layout()
plt.show()

## 3. Model 2: Random Forest Regressor

In [ ]:
rf_model = RandomForestRegressor(n_estimators=200, max_depth=20, min_samples_split=5, min_samples_leaf=2, random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

rf_r2 = r2_score(y_test, rf_preds)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_preds))

print(f'Random Forest - R2: {rf_r2:.4f}, MAE: {rf_mae:.4f}, RMSE: {rf_rmse:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature Importance
sns.barplot(x=rf_model.feature_importances_, y=features, ax=axes[0])
axes[0].set_title('RF: Feature Importances')

# Predicted vs Actual
sns.scatterplot(x=y_test, y=rf_preds, alpha=0.5, ax=axes[1])
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_title('RF: Predicted vs Actual')
axes[1].set_xlabel('Actual Volume')
axes[1].set_ylabel('Predicted Volume')

# Residual Distribution
sns.histplot(y_test - rf_preds, bins=50, kde=True, ax=axes[2])
axes[2].set_title('RF: Residual Distribution')
axes[2].set_xlabel('Residual')

plt.tight_layout()
plt.show()

## 4. Model 3: XGBoost Regressor

In [ ]:
xgb_model = xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=1.0, random_state=42)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

xgb_r2 = r2_score(y_test, xgb_preds)
xgb_mae = mean_absolute_error(y_test, xgb_preds)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))

print(f'XGBoost - R2: {xgb_r2:.4f}, MAE: {xgb_mae:.4f}, RMSE: {xgb_rmse:.4f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature Importance
sns.barplot(x=xgb_model.feature_importances_, y=features, ax=axes[0])
axes[0].set_title('XGB: Feature Importances')

# Predicted vs Actual
sns.scatterplot(x=y_test, y=xgb_preds, alpha=0.5, ax=axes[1])
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_title('XGB: Predicted vs Actual')
axes[1].set_xlabel('Actual Volume')
axes[1].set_ylabel('Predicted Volume')

# Residual Distribution
sns.histplot(y_test - xgb_preds, bins=50, kde=True, ax=axes[2])
axes[2].set_title('XGB: Residual Distribution')
axes[2].set_xlabel('Residual')

plt.tight_layout()
plt.show()

## 5. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest', 'XGBoost'],
    'R2': [lr_r2, rf_r2, xgb_r2],
    'MAE': [lr_mae, rf_mae, xgb_mae],
    'RMSE': [lr_rmse, rf_rmse, xgb_rmse]
})
display(results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x='Model', y='R2', data=results, ax=axes[0])
axes[0].set_title('Model Comparison - R2 Score (Higher is better)')
axes[0].set_ylim(0, 1.0)

sns.barplot(x='Model', y='MAE', data=results, ax=axes[1])
axes[1].set_title('Model Comparison - MAE (Lower is better)')

plt.tight_layout()
plt.show()

**Discussion**: XGBoost and Random Forest significantly outperform Linear Regression, which struggles with the non-linear relationships in crop production across regions and time. The tree-based models achieve higher R² and lower errors. Between RF and XGBoost, the best model can be chosen based on the highest R² score.

## 6. Save Best Model and Export Data

In [ ]:
# Determine best model based on R2
models = {'Linear Regression': lr_model, 'Random Forest': rf_model, 'XGBoost': xgb_model}
best_model_name = results.loc[results['R2'].idxmax(), 'Model']
best_model = models[best_model_name]

print(f'Best model selected: {best_model_name}')

os.makedirs(r'c:\Users\James\Music\CVPP\models', exist_ok=True)
joblib.dump(best_model, r'c:\Users\James\Music\CVPP\models\best_model.joblib')
joblib.dump(le_crop, r'c:\Users\James\Music\CVPP\models\le_crop.joblib')
joblib.dump(le_region, r'c:\Users\James\Music\CVPP\models\le_region.joblib')

os.makedirs(r'c:\Users\James\Music\CVPP\data\processed', exist_ok=True)
df_long.to_csv(r'c:\Users\James\Music\CVPP\data\processed\crops_long_format.csv', index=False)
print('Models and processed data successfully exported.')